In [33]:
using CSV, DataFrames, GLM, StatsPlots, Gurobi, JuMP
using LinearAlgebra, Random, DataFrames, CSV, Plots
using StatsBase, Statistics, Distributions
using JuMP, Gurobi
gurobi_env = Gurobi.Env()

Set parameter Username
Set parameter LicenseID to value 2749630
Academic license - for non-commercial use only - expires 2026-12-03


Gurobi.Env(Ptr{Nothing} @0x000000033ef4c200, false, 0)

In [34]:
# Create a holistic regression model to predict with
#X = CSV.read("../clean_data/train_data_features.csv", DataFrame)
#y = CSV.read("../clean_data/test_data_features.csv", DataFrame)
;

In [36]:
using CSV
using DataFrames
using Statistics

# 1. Read CSV
df = CSV.read("../clean_data/train_data_features.csv", DataFrame)



# 2. Define target (y) and feature set (X)
# HB_NORTH is your response for holistic regression
y = df.HB_NORTH               # Vector{Float64}

# Drop the target and the timestamp from the features
feature_cols = Not([:HB_NORTH, :interval_start_local])
X_df = df[:, feature_cols]    # DataFrame of predictors

# 3. Handle missing values if there are any (simple example: fill with 0.0)
# You can replace 0.0 with mean/median/etc. if you prefer.
X_df = coalesce.(X_df, 0.0)

# 4. Ensure all features are Float64
X = Matrix{Float64}(X_df)     # n × p matrix of features
y_vec = Vector{Float64}(y)    # n-vector of targets

# 5. (Optional but common) Standardize features for regression
X_std = copy(X)
for j in 1:size(X_std, 2)
    μ = mean(X_std[:, j])
    σ = std(X_std[:, j])
    if σ > 0
        X_std[:, j] .= (X_std[:, j] .- μ) ./ σ
    end
end

# Now X_std (or X) and y_vec are ready for holistic regression
# Example (if you're using a HolisticRegression.jl-style API):
# using HolisticRegression
# model = HolisticRegression.fit(X_std, y_vec; λ=..., constraints=...)

In [37]:
size(X_df)

(17362, 128)

In [38]:
# Keep only the first 2,000 rows
X_small = X_std[:, :]
;

In [39]:
size(X_small)

(17362, 128)

In [40]:
# Inspect dataset
describe(X)

Summary Stats:
Length:         2222336
Missing Count:  0
Mean:           4868.743943
Std. Deviation: 12614.691682
Minimum:        -11133.166667
1st Quartile:   0.640000
Median:         29.040000
3rd Quartile:   2243.120000
Maximum:        267824.930000
Type:           Float64


In [41]:
first(X, 5)

5-element Vector{Float64}:
 0.0
 1.0
 2.0
 3.0
 4.0

In [42]:
"""
Holistic regression MIP
"""       
XtX = X' * X
inv_XtX = inv(Symmetric(XtX))   # helps enforce symmetry numerically

diag_inv = diag(inv_XtX)
diag_inv = max.(diag_inv, 0.0)  # clip tiny negatives to 0

function compute_HC(X, ρ_max)
    n,p = size(X)
    c = zeros(p,p)
    for i=1:p-1,j=i+1:p
        c[i,j] = cor(X[:,i],X[:,j])
    end
    return [(i,j) for i=1:p for j=i+1:p if abs(c[i,j])>ρ_max]
end

function compute_sigma(X::Matrix{Float64}, y::Vector{Float64})
    n, p = size(X)
    if n <= p
        error("compute_sigma: need n > p (n = $n, p = $p)")
    end

    # OLS fit in a numerically stable way
    β̂ = X \ y                      # solves min ||Xβ - y||₂
    r  = y - X * β̂                 # residuals

    σ2 = dot(r, r) / (n - p)        # residual variance

    # clip potential tiny negative due to floating point
    σ2 = max(σ2, 0.0)

    return sqrt(σ2)
end
         
function holistic(X::Matrix{Float64}, y::Vector{Float64}, λ::Float64, mu::Float64,
                   ρ_max::Float64, t::Float64)
    M = 50    
    n,p = size(X)
    m = Model(() -> Gurobi.Optimizer(gurobi_env))
    set_optimizer_attribute(m, "OutputFlag", 0)
    #set_optimizer_attribute(m, "TimeLimit", 10000)
    
    @variable(m, β[1:p])
    @variable(m, z[1:p], Bin)
    @variable(m, b[1:p], Bin)    
    @variable(m, s[1:p])
    @variable(m, theta[1:n])

    # Objective
    @objective(m, Min, sum(theta) + λ * sum(s) + mu * sum(β.^2))
    
    @constraint(m, [i in 1:n], theta[i] >= (y[i] - dot(X[i, :], β))) 
    @constraint(m, [i in 1:n], theta[i] >= -(y[i] - dot(X[i, :], β)))   
    
    # 1-norm regularization
    @constraint(m, s .>= β)
    @constraint(m, s .>= -β)

    @constraint(m, [i=1:p], β[i] >= -M*z[i])
    @constraint(m, [i=1:p], β[i] <= M*z[i])

    # Pairwise correlation
    HC = compute_HC(X,ρ_max)
    for (i,j) in HC
        @constraint(m, z[i] + z[j] <= 1)
    end

    # Transformation 
    @constraint(m, [j in 1:4], z[j] + z[6+2(j-1)] + z[6+2(j-1)+1] <= 1)

    optimize!(m)
                    
    return value.(β), value.(z)
end

holistic (generic function with 1 method)

In [43]:
ρ_max = 0.9
t = 1.96
λ = 0.7
mu = 0.3

β2, z2 = holistic(X_std, y, λ, mu, ρ_max, t)

([1.0085218889480136, 0.0, 0.0, -1.0038475940587506, -1.1660065625363127, 0.0, 0.0, 0.0, 0.0, 3.636436951374575  …  0.0, 0.36421480287017105, 0.29997217664595277, 0.0, -0.4346872351457312, 1.6653811775645309, 0.11114199903674969, 0.19741199176649585, 0.24959028072040026, 0.004634217038447698], [1.0, 1.0, 0.0, 1.0, 1.0, 0.0, 0.0, 0.0, -0.0, 1.0  …  0.0, 1.0, 1.0, 0.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0])

In [44]:
# X_test y_test
using CSV
using DataFrames
using Statistics

# 1. Read CSV
df_t = CSV.read("../clean_data/test_data_features.csv", DataFrame)



# 2. Define target (y) and feature set (X)
# HB_NORTH is your response for holistic regression
y_t = df_t.HB_NORTH               # Vector{Float64}

# Drop the target and the timestamp from the features
feature_cols = Not([:HB_NORTH, :interval_start_local])
X_df_t = df_t[:, feature_cols]    # DataFrame of predictors

# 3. Handle missing values if there are any (simple example: fill with 0.0)
# You can replace 0.0 with mean/median/etc. if you prefer.
X_df_t = coalesce.(X_df_t, 0.0)

# 4. Ensure all features are Float64
X_t = Matrix{Float64}(X_df_t)     # n × p matrix of features
y_vec_t = Vector{Float64}(y_t)    # n-vector of targets

# 5. (Optional but common) Standardize features for regression
X_std_t = copy(X_t)
for j in 1:size(X_std_t, 2)
    μ = mean(X_std_t[:, j])
    σ = std(X_std_t[:, j])
    if σ > 0
        X_std_t[:, j] .= (X_std_t[:, j] .- μ) ./ σ
    end
end


In [74]:
function RMSE(y, pred) 
    return sqrt(sum((y-pred).^2)/length(y))
end;

function MAE(y, pred) 
    return mean(abs.((y-pred)))
end;

In [77]:
# Run test mse
test_RMSE = RMSE(y_t, X_std_t * β2)
test_MAE = MAE(y_t, X_std_t * β2)
println(test_RMSE)
println(test_MAE)

60.30080307252483
32.16978493785522


In [73]:
function baseline_day_before_test_rmse(y_test)
    H = 24
    
    # shift by 24 inside the test set itself
    y_pred = [fill(missing, H); y_test[1:end-H]]

    y_true = y_test[(H+1):end]
    y_pred_clean = y_pred[(H+1):end]

    return sqrt(mean((y_true .- y_pred_clean).^2))
end

rmse = baseline_day_before_test_rmse(y_t)
println("Test-set baseline MSE (day-before): ", rmse)

function baseline_day_before_test_mae(y_test)
    H = 24
    
    # shift by 24 inside the test set itself
    y_pred = [fill(missing, H); y_test[1:end-H]]

    y_true = y_test[(H+1):end]
    y_pred_clean = y_pred[(H+1):end]

    return mean(abs.(y_true .- y_pred_clean))
end

mae = baseline_day_before_test_mae(y_t)
println("Test-set baseline MAE (day-before): ", mae)

Test-set baseline MSE (day-before): 67.80580128878344
Test-set baseline MAE (day-before): 19.14169551395267


In [48]:
(rmse - test_RMSE)/rmse

0.11068371840773605

In [49]:
num_selected = count(==(1), z2)
println("Number of selected variables: $num_selected")

Number of selected variables: 71


In [50]:
selected = findall(z2 .== 1)

β2_selected = β2[selected]
abs_selected = abs.(β2_selected)

# Sort by absolute coefficient size
order = sortperm(abs_selected, rev=true)



selected_names = names(X_df)[selected]

println("\nStrongest Features:")
for idx in order
    println(selected_names[idx], ": ", abs_selected[idx])
end


Strongest Features:
load_rolling_24h_hist: 6.231703189554187
weather_HDD65_lag24: 5.6450919195981495
load_system_lag24: 4.332878947740732
load_system_lag4: 4.208656670021816
weather_HDD65_lag1: 3.636436951374575
solar_southeast_lag12: 3.6273306593980714
wind_lz_west_lag1: 2.845572769696871
weather_slp_hPa_lag24: 2.7941607067548664
weather_dewpoint_C_lag24: 2.4500016857742013
solar_centerwest_lag12: 2.3891966101904885
wind_lz_south_houston_lag1: 2.2485617694726576
wind_lz_west_lag4: 2.094970595006941
solar_centereast_lag24: 1.7166706669504628
solar_northwest_lag1: 1.6928387463875048
price_lag1: 1.6653811775645309
solar_fareast_lag1: 1.587644477371917
solar_southeast_lag2: 1.564197554686839
weather_HDD65_lag12: 1.5545079559562842
wind_lz_north_lag1: 1.5320899709716527
wind_lz_south_houston_lag4: 1.531442529995912
wind_lz_north_lag4: 1.5276812987998172
wind_lz_north_lag12: 1.5265609151978097
weather_CDD65_lag4: 1.511014243781451
weather_temp_f_lag24: 1.4196293945483092
solar_southeast_la

In [54]:
using Pkg
Pkg.add("GLPK")

    Updating registry at `~/.julia/registries/General.toml`
   Resolving package versions...
   Installed GLPK_jll ─ v5.0.1+1
   Installed GLPK ───── v1.2.1
    Updating `~/.julia/environments/v1.11/Project.toml`
  [60bf3e95] + GLPK v1.2.1
    Updating `~/.julia/environments/v1.11/Manifest.toml`
  [60bf3e95] + GLPK v1.2.1
  [e8aa6df9] + GLPK_jll v5.0.1+1
  [781609d7] + GMP_jll v6.3.0+0
Precompiling project...
   2171.9 ms  ✓ GMP_jll
   1305.2 ms  ✓ GLPK_jll
   2949.7 ms  ✓ GLPK
  3 dependencies successfully precompiled in 7 seconds. 157 already precompiled.


In [57]:
using JuMP
using GLPK  # or your favorite solver

"""
    quantile_regression(X, y; tau=0.5)

Solve quantile regression for given tau (median when tau=0.5).
X: Matrix (n, p)
y: Vector length n
"""
function quantile_regression(X, y; tau=0.5)
    n, p = size(X)
    X_design = hcat(ones(n), X)  # (n, p+1)
    p_full = p + 1

    model = Model(GLPK.Optimizer)

    @variable(model, β[1:p_full])
    @variable(model, r_pos[1:n] >= 0)  # max(residual, 0)
    @variable(model, r_neg[1:n] >= 0)  # max(-residual, 0)

    # residual: y - xᵢ'β = r_pos[i] - r_neg[i]
    @constraint(model, [i in 1:n],
        y[i] - sum(X_design[i, j] * β[j] for j in 1:p_full) == r_pos[i] - r_neg[i]
    )

    @objective(model, Min,
        tau * sum(r_pos[i] for i in 1:n) +
        (1 - tau) * sum(r_neg[i] for i in 1:n)
    )

    optimize!(model)

    return value.(β), objective_value(model)
end

# Example:
# β_opt, obj = quantile_regression(X, y; tau=0.5)  # median regression
# println("Intercept: ", β_opt[1])
# println("Slopes: ", β_opt[2:end])

quantile_regression

In [68]:
B_qr, obj_qr = quantile_regression(X_std, y, tau=0.25)

([18.129408256778227, -0.11769963887300221, 0.04641307981560405, -0.0357763060516995, 0.009508864445868744, 0.8313946030876506, 0.0, 0.4847827467538864, 1.008207154828578, -1.4098136897007298  …  0.0, 0.0, 0.781548144084794, 4.865377080140897, -5.049166385191388, 13.991287179963404, 0.5481893460713525, -0.3364545407950328, -0.10627170407636555, 1.5089855717931757], 92284.84067312667)

In [69]:
n_test, p = size(X_std_t)

# Match what we did in training
X_test_design = hcat(ones(n_test), X_std_t)  # (n_test, p+1)



5686×129 Matrix{Float64}:
 1.0  -1.66141   -0.00166817  …  -0.2452      0.836345   -0.218195
 1.0  -1.51694   -0.00166817     -0.249085    1.50788    -0.191389
 1.0  -1.37247   -0.00166817     -0.234125    0.689295   -0.163806
 1.0  -1.228     -0.00166817     -0.224217    0.0426238  -0.332995
 1.0  -1.08354   -0.00166817     -0.155246   -0.112002   -0.462946
 1.0  -0.939067  -0.00166817  …  -0.165931   -0.185236   -0.361938
 1.0  -0.794599  -0.00166817     -0.117749   -0.24021    -0.319009
 1.0  -0.650131  -0.00166817     -0.279588   -0.237879   -0.317456
 1.0  -0.505663  -0.00166817     -0.178754   -0.245066   -0.32542
 1.0  -0.361195  -0.00166817     -0.175451   -0.248951   -0.488004
 ⋮                            ⋱                          
 1.0   0.505613  -1.00011         0.59761    -0.0117673   0.745852
 1.0   0.650081  -1.00011         0.159497   -0.158623    0.00635416
 1.0   0.794549  -1.00011         0.55681    -0.186207   -0.028416
 1.0   0.939017  -1.00011     …   0.453839  

In [78]:
# Run test mse
test_RMSE_qr = RMSE(y_t, X_test_design * B_qr)
test_MAE_qr = MAE(y_t, X_test_design * B_qr)
println(test_RMSE_qr)
println(test_MAE_qr)

49.96601786167413
17.12128889485579


In [81]:
test_prices = X_test_design * B_qr;

In [83]:
test_prices

5686-element Vector{Float64}:
  7.751260420385692
  9.976205264286673
 10.803492074660829
 11.727772642441982
  8.87475135939946
  8.768133100117934
  6.9667875639721375
  8.821232854275502
  9.302607373585845
  7.658639360840268
  ⋮
 31.584059929504583
 30.82729536080087
 36.82378000507497
 29.96482430239693
 32.57878433934019
 35.120928761922464
 35.51248703412355
 34.563826314989115
 36.676794329333575

In [86]:
using CSV, DataFrames


df = DataFrame(value = test_prices)   # wrap vector in a one-column DataFrame
CSV.write("../price_data/vector_out.csv", df)

"../price_data/vector_out.csv"